In [2]:
# 시각화 및 지도 시각화

import pandas as pd
import numpy as np
import folium
from folium import plugins
import json

# 1. 데이터 로드
file_path = r'C:\py_temp_5\사람인\훈련용\posting_analysis_table_최종.xlsx'
df = pd.read_excel(file_path)

# 2. 지역별 통계 집계 함수 (변경 없음)
def get_regional_stats(data):
    if data.empty: return pd.DataFrame()
    stats = data.groupby('시각화용_지역').agg({
        'posting_quality_score': 'mean',
        'risk_signal_score': 'mean',
        '시각화용_지역': 'count'
    }).rename(columns={'시각화용_지역': '공고수'}).reset_index()
    stats['geo_name'] = stats['시각화용_지역'].apply(lambda x: x.split(' ')[-1])
    stats['log_count'] = np.log1p(stats['공고수'])
    return stats

df_total = get_regional_stats(df)
df_da = get_regional_stats(df[df['job'] == '데이터 분석가'])
df_be = get_regional_stats(df[df['job'] == '백엔드 개발자'])

# 3. GeoJSON 로드 (인코딩 명시 권장)
with open(r'C:\py_temp_5\사람인\skorea-municipalities-2018-geo.json', encoding='utf-8') as f:
    geo_data = json.load(f)

# 4. 지도 초기 객체 생성
m = folium.Map(location=[37.56, 126.97], zoom_start=10, tiles='cartodbpositron')

# 5. 시각화 레이어 생성 함수 (구조 개선)
def add_choropleth_layer(map_obj, stats_df, layer_name, color_palette):
    if stats_df.empty: return

    # 중복 지역명 처리
    mapping_df = stats_df.drop_duplicates(subset=['geo_name'], keep='first')
    info_dict = mapping_df.set_index('geo_name').to_dict(orient='index')

    # [핵심] Choropleth를 지도에 직접 넣지 않고, 색상 매핑용 데이터만 활용합니다.
    cp = folium.Choropleth(
        geo_data=geo_data,
        data=mapping_df,
        columns=['geo_name', 'log_count'],
        key_on='feature.properties.name',
        fill_color=color_palette,
        fill_opacity=0.7,
        line_opacity=0.3,
        nan_fill_color='#F2F2F2',
        nan_fill_opacity=0.5,
        legend_name=f'{layer_name} (Log Scale)'
    )
    
    # 1. 범례(Legend)만 지도에 추가
    map_obj.add_child(cp.color_scale)

    # 2. GeoJson 객체에 데이터 및 툴팁 주입
    geojson_layer = cp.geojson
    for feature in geojson_layer.data['features']:
        region_name = feature['properties']['name']
        data = info_dict.get(region_name)
        
        if data:
            feature['properties']['description'] = (
                f"<b>[{region_name}]</b><br>"
                f"공고수: {int(data['공고수'])}건<br>"
                f"품질 점수: {data['posting_quality_score']:.1f}점<br>"
                f"위험 신호: {data['risk_signal_score']:.1%}"
            )
        else:
            feature['properties']['description'] = f"<b>[{region_name}]</b><br>공고 데이터 없음"

    # 3. 툴팁 설정
    folium.GeoJsonTooltip(
        fields=['description'],
        aliases=[''],
        labels=False,
        sticky=True
    ).add_to(geojson_layer)

    # 4. FeatureGroup에 담아서 지도에 추가 (토글용)
    fg = folium.FeatureGroup(name=layer_name)
    geojson_layer.add_to(fg)
    fg.add_to(map_obj)

# 6. 직무별 레이어 추가 (토글 메뉴 구성)
add_choropleth_layer(m, df_total, '전체 공고', 'Blues')
add_choropleth_layer(m, df_da, '데이터 분석가', 'Purples')
add_choropleth_layer(m, df_be, '백엔드 개발자', 'Oranges')

# 7. 부가 기능 추가
folium.LayerControl(collapsed=False).add_to(m) # 레이어 선택기
plugins.Fullscreen(position='topright').add_to(m) # 전체화면 버튼

# 8. HTML 저장
output_name = r'C:\py_temp_5\사람인\it_job_market_map.html'
m.save(output_name)
print(f"시각화 완료: {output_name} 파일을 확인하세요.")

시각화 완료: C:\py_temp_5\사람인\it_job_market_map.html 파일을 확인하세요.


In [1]:
import pandas as pd
import numpy as np
import folium
import json

# 1. 데이터 로드
file_path = r'C:\py_temp_5\사람인\훈련용\posting_analysis_table_최종.xlsx'
df = pd.read_excel(file_path)

# 2. 지역별 통계 집계 (원본 이름 유지)
stats = df.groupby('시각화용_지역').agg({
    'posting_quality_score': 'mean',
    'risk_signal_score': 'mean',
    'job': 'count'
}).rename(columns={'job': '공고수'}).reset_index()

# 3. GeoJSON 로드
with open(r'C:\py_temp_5\사람인\skorea-municipalities-2018-geo.json', encoding='utf-8') as f:
    geo_data = json.load(f)

# 4. [핵심] 유연한 매칭 로직 (지도의 이름을 기준으로 데이터 찾기)
# 엑셀의 '서울 강서구'를 {'서울', '강서구'} 세트로 만들어 비교합니다.
stats_list = stats.to_dict(orient='records')
for item in stats_list:
    item['name_set'] = set(item['시각화용_지역'].split())

# 지도 데이터의 각 구역(feature)을 돌며 데이터 주입
for feature in geo_data['features']:
    geo_name = feature['properties']['name'] # 지도상 이름 (예: "서울특별시 강서구" 또는 "성남시분당구")
    
    match_data = None
    # 엑셀 데이터 중 해당 지명과 겹치는 단어가 가장 많은 것을 찾음
    for s in stats_list:
        # 예: '서울', '강서구' 두 단어가 '서울특별시 강서구' 안에 다 있는지 확인
        if all(word in geo_name or geo_name.startswith(word) for word in s['name_set']):
            match_data = s
            break
        # 경기도 특수 처리 (성남시분당구 등)
        elif '경기' in s['name_set'] and any(word in geo_name for word in s['name_set'] if word != '경기'):
            match_data = s
            break

    if match_data:
        feature['properties']['match_val'] = np.log1p(match_data['공고수'])
        feature['properties']['description'] = (
            f"<b>[{geo_name}]</b><br>"
            f"매칭된 데이터: {match_data['시각화용_지역']}<br>"
            f"공고수: {match_data['공고수']}건<br>"
            f"품질: {match_data['posting_quality_score']:.1f}점<br>"
            f"위험: {match_data['risk_signal_score']:.1%}"
        )
    else:
        feature['properties']['match_val'] = 0
        feature['properties']['description'] = f"<b>[{geo_name}]</b><br>데이터 매칭 실패"

# 5. 지도 생성
m = folium.Map(location=[37.56, 126.97], zoom_start=10, tiles='cartodbpositron')

# 컬러 스케일 생성
max_val = max([f['properties'].get('match_val', 0) for f in geo_data['features']])
color_scale = folium.branca.colormap.linear.YlGnBu_09.scale(0, max_val)
color_scale.caption = '공고수 분포 (Log Scale)'

def style_fn(feature):
    val = feature['properties'].get('match_val', 0)
    return {
        'fillColor': color_scale(val) if val > 0 else '#F2F2F2',
        'color': 'gray',
        'weight': 0.5,
        'fillOpacity': 0.7 if val > 0 else 0.3
    }

# 지도 레이어 추가
folium.GeoJson(
    geo_data,
    style_function=style_fn,
    tooltip=folium.GeoJsonTooltip(fields=['description'], aliases=[''], labels=False)
).add_to(m)

m.add_child(color_scale)
m.save(r'C:\py_temp_5\사람인\final_matching_success.html')
print("매칭 시도 완료! HTML을 확인하세요.")

매칭 시도 완료! HTML을 확인하세요.


In [3]:
import pandas as pd
import numpy as np
import folium
import json
from folium import plugins

# 1. 데이터 로드
file_path = r'C:\py_temp_5\사람인\훈련용\posting_analysis_table_최종.xlsx'
df = pd.read_excel(file_path)

# [추가] 지명 전처리 함수: 매칭 확률을 높이기 위해 불필요한 수식어 제거
def clean_region_name(name):
    # '특별시', '광역시', '특별자치도' 등 제거하여 비교용 텍스트 생성
    removals = ['특별시', '광역시', '특별자치도', '특별자치시', '경기도', '충청북도', '충청남도', '전라북도', '전라남도', '경상북도', '경상남도', '강원도', '제주도']
    for r in removals:
        name = name.replace(r, r[:2]) # '경상남도' -> '경남'
    return name.replace(" ", "")

# 2. 직무별 데이터 분리 및 통계 집계
jobs = ['데이터 분석', '백엔드 개발자']
job_stats = {}

for job_name in jobs:
    # 해당 직무 데이터만 필터링 (컬럼명 'job' 기준)
    filtered_df = df[df['job'].str.contains(job_name, na=False)]
    
    stats = filtered_df.groupby('시각화용_지역').agg({
        'posting_quality_score': 'mean',
        'risk_signal_score': 'mean',
        'job': 'count'
    }).rename(columns={'job': '공고수'}).reset_index()
    
    # 매칭용 세트 생성
    stats_list = stats.to_dict(orient='records')
    for item in stats_list:
        item['clean_name'] = clean_region_name(item['시각화용_지역'])
    
    job_stats[job_name] = stats_list

# 3. GeoJSON 로드
with open(r'C:\py_temp_5\사람인\skorea-municipalities-2018-geo.json', encoding='utf-8') as f:
    geo_data_raw = json.load(f)

# 4. 지도 생성 및 레이어 추가 함수
m = folium.Map(location=[36.5, 127.5], zoom_start=7, tiles='cartodbpositron')

def create_job_layer(job_name, data_list, geo_json_base):
    # 각 직무별로 독립적인 GeoJSON 객체 생성
    import copy
    current_geo = copy.deepcopy(geo_json_base)
    
    for feature in current_geo['features']:
        geo_name = feature['properties']['name']
        clean_geo = clean_region_name(geo_name)
        
        match_data = None
        # 개선된 매칭 로직: 데이터의 지역명이 지도 지명에 포함되거나 그 반대인 경우
        for s in data_list:
            if s['clean_name'] in clean_geo or clean_geo in s['clean_name']:
                match_data = s
                break
        
        if match_data:
            feature['properties']['match_val'] = np.log1p(match_data['공고수'])
            feature['properties']['description'] = (
                f"<b>[{geo_name}] ({job_name})</b><br>"
                f"공고수: {match_data['공고수']}건<br>"
                f"평균 품질: {match_data['posting_quality_score']:.1f}점<br>"
                f"위험 신호: {match_data['risk_signal_score']:.1%}"
            )
        else:
            feature['properties']['match_val'] = 0
            feature['properties']['description'] = f"<b>[{geo_name}]</b><br>데이터 없음"

    # 컬러 스케일 (직무별로 동적 생성하거나 공통 사용)
    layer = folium.FeatureGroup(name=job_name)
    
    folium.GeoJson(
        current_geo,
        style_function=lambda x: {
            'fillColor': '#YlOrRd' if job_name == '데이터 분석' else '#BuPu', # 직무별 색상 차별화 가능
            'fillColor': folium.branca.colormap.linear.YlGnBu_09(x['properties']['match_val']) 
                         if x['properties']['match_val'] > 0 else '#F2F2F2',
            'color': 'gray', 'weight': 0.5, 'fillOpacity': 0.7
        },
        tooltip=folium.GeoJsonTooltip(fields=['description'], labels=False)
    ).add_to(layer)
    
    return layer

# 5. 직무별 레이어 추가
for job_name in jobs:
    layer = create_job_layer(job_name, job_stats[job_name], geo_data_raw)
    m.add_child(layer)

# 레이어 컨트롤 추가
folium.LayerControl(collapsed=False).add_to(m)

m.save(r'C:\py_temp_5\사람인\job_analysis_comparison.html')
print("분석 완료! 직무별 레이어를 확인하세요.")

분석 완료! 직무별 레이어를 확인하세요.


In [8]:
import pandas as pd
import numpy as np
import folium
import json
import copy

# 1. 데이터 로드
file_path = r'C:\py_temp_5\사람인\훈련용\posting_analysis_table_최종.xlsx'
df = pd.read_excel(file_path)

# [수정] 엑셀용 지역명 정규화 (시도 앞2글자 + 시군구 공백제거)
def normalize_region_excel(raw_name):
    parts = str(raw_name).split()
    if len(parts) < 2: return str(raw_name)[:2] # 세종 등
    
    sido = parts[0][:2] # '서울', '경기', '경남'
    sigungu = parts[1]
    
    # 경기도 대도시 통합 (수원시 장안구 -> 수원시)
    big_cities = ['수원', '성남', '안양', '안산', '용인', '고양']
    for city in big_cities:
        if sigungu.startswith(city):
            sigungu = city + "시"
            break
    return f"{sido} {sigungu}".replace(" ", "") # '경기성남시' 형태로 공백 제거

# 2. 직무별 데이터 집계
job_map = {
    '전체': df,
    '데이터 분석': df[df['job'].str.contains('데이터 분석', na=False)],
    '백엔드 개발자': df[df['job'].str.contains('백엔드 개발자', na=False)]
}

job_results = {}
for label, target_df in job_map.items():
    tdf = target_df.copy()
    tdf['match_key'] = tdf['시각화용_지역'].apply(normalize_region_excel)
    
    stats = tdf.groupby('match_key').agg({
        'posting_quality_score': 'mean',
        'risk_signal_score': 'mean',
        'job': 'count'
    }).rename(columns={'job': '공고수'}).reset_index()
    
    job_results[label] = stats.set_index('match_key').to_dict(orient='index')

# 3. GeoJSON 로드
with open(r'C:\py_temp_5\사람인\skorea-municipalities-2018-geo.json', encoding='utf-8') as f:
    geo_data = json.load(f)

# 4. 지도 설정
m = folium.Map(location=[36.5, 127.5], zoom_start=7, tiles='cartodbpositron')

# 컬러 스케일 (전체 데이터 기준으로 범위 설정)
all_counts = [v['공고수'] for v in job_results['전체'].values()]
max_log = np.log1p(max(all_counts)) if all_counts else 1
color_scale = folium.branca.colormap.linear.YlGnBu_09.scale(0, max_log)

# 5. [수정] 레이어 생성 함수 (GeoJSON 키 생성 로직 강화)
def create_job_layer(display_name, data_dict):
    layer = folium.FeatureGroup(name=display_name, overlay=False)
    curr_geo = copy.deepcopy(geo_data)
    
    for feature in curr_geo['features']:
        p = feature['properties']
        full_name = p.get('name', '') # 예: "경기도 성남시 분당구" 또는 "서울특별시 강서구"
        
        parts = full_name.split()
        if len(parts) >= 2:
            g_sido = parts[0][:2]
            g_sigungu = parts[1]
            
            # 경기도 대도시 통합
            for city in ['수원', '성남', '안양', '안산', '용인', '고양']:
                if g_sigungu.startswith(city):
                    g_sigungu = city + "시"
                    break
            g_key = f"{g_sido}{g_sigungu}" # 공백 없이 '경기성남시'
        else:
            g_key = full_name[:2] # 세종 등
        
        match = data_dict.get(g_key)
        
        if match:
            val = np.log1p(match['공고수'])
            tooltip_text = (
                f"<b>[{full_name}]</b><br>공고: {int(match['공고수'])}건<br>"
                f"품질: {match['posting_quality_score']:.1f} / 위험: {match['risk_signal_score']:.1%}"
            )
        else:
            val = 0
            tooltip_text = f"<b>[{full_name}]</b><br>데이터 없음 (키: {g_key})" # 디버깅용 키 표시
            
        p['match_val'] = val
        p['tooltip'] = tooltip_text

    folium.GeoJson(
        curr_geo,
        style_function=lambda x: {
            'fillColor': color_scale(x['properties']['match_val']) if x['properties']['match_val'] > 0 else '#F2F2F2',
            'color': 'white', 'weight': 0.5, 'fillOpacity': 0.7
        },
        tooltip=folium.GeoJsonTooltip(fields=['tooltip'], labels=False)
    ).add_to(layer)
    return layer

# 6. 레이어 추가 및 실행
for label in ['전체', '데이터 분석', '백엔드 개발자']:
    m.add_child(create_job_layer(label, job_results[label]))

folium.LayerControl(collapsed=False).add_to(m)
color_scale.add_to(m)

m.save(r'C:\py_temp_5\사람인\final_fixed_map.html')
print("지도 생성 완료! 만약 회색이라면 툴팁에 뜨는 (키: XXX)를 확인하세요.")

지도 생성 완료! 만약 회색이라면 툴팁에 뜨는 (키: XXX)를 확인하세요.
